In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

print("=" * 60)
print("FASE 1: INGESTÃO CSV → DELTA LAKE")
print("=" * 60)

# Parâmetros do projeto
VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi"
CSV_PATH = f"{VOLUME_PATH}/raw/csv"
DELTA_PATH = f"{VOLUME_PATH}/raw/delta"

# Arquivos do dataset (nomes reais do Kaggle)
FILES = ["2015-01", "2016-01", "2016-02", "2016-03"]

print(f"\n📁 Volume: {VOLUME_PATH}")
print(f"📁 CSVs: {CSV_PATH}")
print(f"📁 Delta: {DELTA_PATH}")

In [0]:
# TAREFA 1.2: Verificar arquivos salvos
print("\n" + "="*60)
print("TAREFA 1.2: Verificando CSVs no Volume")
print("="*60)

csv_files = dbutils.fs.ls(CSV_PATH)

total_size_gb = 0
for file in csv_files:
    size_gb = file.size / 1e9
    total_size_gb += size_gb
    print(f"✅ {file.name} - {size_gb:.2f} GB")

print(f"\n📊 Total: {total_size_gb:.2f} GB em {len(csv_files)} arquivos")

In [0]:
# TAREFA 1.3: Exploração do Schema
print("\n" + "="*60)
print("TAREFA 1.3: Exploração do schema (2016-01)")
print("="*60)

df_sample = spark.read.csv(
    f"{CSV_PATH}/yellow_tripdata_2016-01.csv",
    header=True,
    inferSchema=True
)

print(f"\n📊 Schema do Dataset:")
df_sample.printSchema()

print(f"\n📈 Contagem de linhas: {df_sample.count():,}")

print(f"\n📋 Primeiras 5 linhas:")
display(df_sample.limit(5))

In [0]:
# TAREFA 1.4: Conversão para Delta Lake
print("\n" + "="*60)
print("TAREFA 1.4: Conversão para Delta Lake")
print("="*60)

resultados = []

for periodo in FILES:
    try:
        print(f"\n🔄 Processando {periodo}...")
        
        # 1. Ler CSV
        csv_file = f"{CSV_PATH}/yellow_tripdata_{periodo}.csv"
        df = spark.read.csv(csv_file, header=True, inferSchema=True)
        
        # 2. Limpeza mínima (Viagens com distância zero, Tarifas zeradas ou negativas e Registros sem data de início)
        df = df.filter(col("trip_distance") > 0) \
                .filter(col("fare_amount") > 0) \
                .filter(col("tpep_pickup_datetime").isNotNull())
        
        # 3. Extrair data para particionamento
        df = df.withColumn(
            "pickup_date",
            to_date(col("tpep_pickup_datetime"))
        )
        
        # 4. Salvar como Delta particionado
        delta_path = f"{DELTA_PATH}/taxi_{periodo}"
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .partitionBy("pickup_date") \
            .save(delta_path)
        
        row_count = df.count()
        resultados.append({"periodo": periodo, "linhas": row_count, "status": "OK"})
        print(f"✅ {periodo}: {row_count:,} linhas salvas em Delta")
        
    except Exception as e:
        resultados.append({"periodo": periodo, "linhas": 0, "status": f"ERRO: {str(e)}"})
        print(f"❌ Erro ao processar {periodo}: {str(e)}")

print("\n" + "="*60)
print("✅ PROCESSAMENTO CONCLUÍDO!")
print("="*60)
for r in resultados:
    print(f"  {r['periodo']}: {r['linhas']:,} linhas — {r['status']}")

In [0]:
# TAREFA 1.5: Validação
print("\n" + "="*60)
print("TAREFA 1.5: Validação dos dados Delta")
print("="*60)

for periodo in FILES:
    delta_path = f"{DELTA_PATH}/taxi_{periodo}"
    df = spark.read.format("delta").load(delta_path)
    
    row_count = df.count()
    dias_distintos = df.select("pickup_date").distinct().count()
    
    print(f"✅ {periodo}: {row_count:,} linhas, {dias_distintos} dias distintos (partições por data)")

In [0]:
# TAREFA 1.6: Limpeza - Deletar CSVs
print("\n" + "="*60)
print("TAREFA 1.6: Deletando CSVs originais")
print("="*60)

csv_files = dbutils.fs.ls(CSV_PATH)
deleted_size = 0

for file in csv_files:
    if file.name.endswith(".csv"):
        dbutils.fs.rm(file.path)
        deleted_size += file.size
        print(f"🗑️  Deletado: {file.name}")

print(f"\n✅ Liberado: {deleted_size / 1e9:.2f} GB de espaço")

In [0]:
import os

def get_delta_size_gb(path):
    total_bytes = 0
    for item in dbutils.fs.ls(path):
        if item.isDir():
            total_bytes += get_delta_size_gb(item.path) * 1e9
        else:
            total_bytes += item.size
    return total_bytes / 1e9

for periodo in FILES:
    delta_path = f"{DELTA_PATH}/taxi_{periodo}"
    size = get_delta_size_gb(delta_path)
    print(f"📦 {periodo}: {size:.2f} GB")